# Notebook 5: OpenStreetMap Feature Extraction

### Objective

The objective of this notebook is to enrich the retail intelligence project with geospatial features obtained from OpenStreetMap (OSM). Instead of processing large nationwide GIS files, this notebook retrieves only the relevant geographic information required for retail expansion analysis.

The extracted features, including road networks, commercial areas, residential zones, hospitals, schools, shopping malls, and public transport facilities, will later be integrated with Census data and dark store locations to improve the recommendation engine.

In [ ]:
!pip install -q osmnx geopandas

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 104.7/104.7 kB 3.2 MB/s eta 0:00:00


In [ ]:
import pandas as pd
import numpy as np
import osmnx as ox
import geopandas as gpd
import time
import os

In [ ]:
from google.colab import files

uploaded = files.upload()

Saving clean_darkstores.csv to clean_darkstores (2).csv


In [ ]:
# Load Dark Store Dataset

stores = pd.read_csv("clean_darkstores.csv")

print("Total Stores:", len(stores))

stores.head()

Total Stores: 4081


,id,accuracy,latitude,longitude,brand,name,city,state,locality
0,42985,43.0,8.478260,76.954711,Blinkit,NaN,NaN,NaN,NaN
1,40038,13.0,8.525199,76.955395,Blinkit,NaN,NaN,NaN,NaN
2,38514,30.0,9.587031,76.535782,Blinkit,NaN,NaN,NaN,NaN
3,38406,48.0,9.922391,78.095686,Blinkit,NaN,NaN,NaN,NaN
4,39056,0.0,9.949993,76.253442,Blinkit,NaN,NaN,NaN,NaN


In [ ]:
# Validate Store Coordinates

valid_stores = stores[
    stores["latitude"].between(6, 38) &
    stores["longitude"].between(68, 98)
].copy()

valid_stores.reset_index(drop=True, inplace=True)

print("Original Stores:", len(stores))
print("Valid Stores:", len(valid_stores))
print("Removed Stores:", len(stores) - len(valid_stores))

Original Stores: 4081
Valid Stores: 4081
Removed Stores: 0


In [ ]:
# Select Test Store

test_store = valid_stores.iloc[0]

test_lat = test_store["latitude"]
test_lon = test_store["longitude"]

print("Brand:", test_store["brand"])
print("Latitude:", test_lat)
print("Longitude:", test_lon)

Brand: Blinkit
Latitude: 8.478259561
Longitude: 76.95471091


In [ ]:
# Define OSM Features

osm_tags = {
    "amenity": ["school", "college", "hospital"],
    "shop": ["mall"],
    "public_transport": True,
    "landuse": ["commercial", "residential"]
}

In [ ]:
# Extract Nearby OSM Features

test_osm = ox.features_from_point(
    (test_lat, test_lon),
    tags=osm_tags,
    dist=1000
)

print("OSM Connection Successful")
print("Features Found:", len(test_osm))

OSM Connection Successful
Features Found: 105


In [ ]:
# Count Nearby Features

feature_counts = {
    "nearby_schools": 0,
    "nearby_colleges": 0,
    "nearby_hospitals": 0,
    "nearby_malls": 0,
    "public_transport_points": 0,
    "commercial_areas": 0,
    "residential_areas": 0
}

if "amenity" in test_osm.columns:
    feature_counts["nearby_schools"] = int(
        (test_osm["amenity"] == "school").sum()
    )

    feature_counts["nearby_colleges"] = int(
        (test_osm["amenity"] == "college").sum()
    )

    feature_counts["nearby_hospitals"] = int(
        (test_osm["amenity"] == "hospital").sum()
    )

if "shop" in test_osm.columns:
    feature_counts["nearby_malls"] = int(
        (test_osm["shop"] == "mall").sum()
    )

if "public_transport" in test_osm.columns:
    feature_counts["public_transport_points"] = int(
        test_osm["public_transport"].notna().sum()
    )

if "landuse" in test_osm.columns:
    feature_counts["commercial_areas"] = int(
        (test_osm["landuse"] == "commercial").sum()
    )

    feature_counts["residential_areas"] = int(
        (test_osm["landuse"] == "residential").sum()
    )

feature_counts

{'nearby_schools': 23,
 'nearby_colleges': 0,
 'nearby_hospitals': 13,
 'nearby_malls': 1,
 'public_transport_points': 52,
 'commercial_areas': 13,
 'residential_areas': 3}

In [ ]:
# Create OSM Feature Dataset

osm_proof = pd.DataFrame([{
    "brand": test_store["brand"],
    "latitude": test_lat,
    "longitude": test_lon,
    **feature_counts
}])

osm_proof

,brand,latitude,longitude,nearby_schools,nearby_colleges,nearby_hospitals,nearby_malls,public_transport_points,commercial_areas,residential_areas
0,Blinkit,8.47826,76.954711,23,0,13,1,52,13,3


In [ ]:
# Create Neighborhood Activity Score

osm_proof["neighborhood_activity_score"] = (
    osm_proof["nearby_schools"] +
    osm_proof["nearby_colleges"] +
    osm_proof["nearby_hospitals"] +
    osm_proof["nearby_malls"] +
    osm_proof["public_transport_points"] +
    osm_proof["commercial_areas"] +
    osm_proof["residential_areas"]
)

osm_proof

,brand,latitude,longitude,nearby_schools,nearby_colleges,nearby_hospitals,nearby_malls,public_transport_points,commercial_areas,residential_areas,neighborhood_activity_score
0,Blinkit,8.47826,76.954711,23,0,13,1,52,13,3,105


In [ ]:
# Display OSM Analysis

print("Brand:", osm_proof.loc[0, "brand"])
print("Schools within 1 KM:", osm_proof.loc[0, "nearby_schools"])
print("Colleges within 1 KM:", osm_proof.loc[0, "nearby_colleges"])
print("Hospitals within 1 KM:", osm_proof.loc[0, "nearby_hospitals"])
print("Malls within 1 KM:", osm_proof.loc[0, "nearby_malls"])
print("Public Transport Points:", osm_proof.loc[0, "public_transport_points"])
print("Commercial Areas:", osm_proof.loc[0, "commercial_areas"])
print("Residential Areas:", osm_proof.loc[0, "residential_areas"])
print(
    "Neighborhood Activity Score:",
    osm_proof.loc[0, "neighborhood_activity_score"]
)

Brand: Blinkit
Schools within 1 KM: 23
Colleges within 1 KM: 0
Hospitals within 1 KM: 13
Malls within 1 KM: 1
Public Transport Points: 52
Commercial Areas: 13
Residential Areas: 3
Neighborhood Activity Score: 105


In [ ]:
# Save OSM Proof of Concept

osm_proof.to_csv(
    "osm_proof_of_concept.csv",
    index=False
)

print("osm_proof_of_concept.csv created successfully.")

osm_proof_of_concept.csv created successfully.


In [ ]:
# Download OSM Dataset

from google.colab import files

files.download("osm_proof_of_concept.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
# Balanced sample for OSM analysis

sample_stores = (
    valid_stores
    .groupby("brand", group_keys=False)
    .apply(lambda x: x.sample(
        min(10, len(x)),
        random_state=42
    ))
    .reset_index(drop=True)
)

print(sample_stores["brand"].value_counts())

brand
Blinkit             10
Swiggy Instamart    10
Zepto               10
Name: count, dtype: int64


/tmp/ipython-input-38-2780072675.py:6: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda x: x.sample(


In [ ]:
import plotly.graph_objects as go
import pandas as pd

# ==========================================
# PREPARE OSM ENVIRONMENT DATA
# ==========================================

features = {
    "Schools": "nearby_schools",
    "Colleges": "nearby_colleges",
    "Hospitals": "nearby_hospitals",
    "Malls": "nearby_malls",
    "Public Transport": "public_transport_points",
    "Commercial Areas": "commercial_areas",
    "Residential Areas": "residential_areas"
}

env = pd.DataFrame({
    "Environment": list(features.keys()),
    "Count": [
        osm_proof[col].iloc[0]
        for col in features.values()
    ]
})

env = env.sort_values("Count", ascending=True)


# ==========================================
# FIGURE
# ==========================================

fig = go.Figure()

# Background track
for _, row in env.iterrows():

    fig.add_trace(
        go.Scatter(
            x=[0, row["Count"]],
            y=[row["Environment"], row["Environment"]],

            mode="lines",

            line=dict(
                width=8,
                color="#E6ECEF"
            ),

            hoverinfo="skip",
            showlegend=False
        )
    )


# Main location markers
fig.add_trace(
    go.Scatter(

        x=env["Count"],
        y=env["Environment"],

        mode="markers+text",

        marker=dict(
            size=23,

            color=env["Count"],

            colorscale=[
                [0.00, "#D9F0E8"],
                [0.35, "#78C6B0"],
                [0.70, "#347C98"],
                [1.00, "#313695"]
            ],

            line=dict(
                color="white",
                width=2
            ),

            showscale=False
        ),

        text=env["Count"],
        textposition="middle right",

        textfont=dict(
            size=13,
            color="#263B5E"
        ),

        hovertemplate=(
            "<b>%{y}</b><br>"
            "%{x} features within 1 km"
            "<extra></extra>"
        ),

        showlegend=False
    )
)


# ==========================================
# STORE DETAILS
# ==========================================

brand = osm_proof["brand"].iloc[0]

score = osm_proof[
    "neighborhood_activity_score"
].iloc[0]


# ==========================================
# DESIGN
# ==========================================

fig.update_layout(

    title=dict(
        text=(
            "<b>MICRO-LOCATION ENVIRONMENT PROFILE</b><br>"
            "<span style='font-size:13px;color:#777777'>"
            f"{brand} dark store • Infrastructure within 1 km "
            f"• Neighborhood Activity Score: {score}"
            "</span>"
        ),

        x=0.5,
        xanchor="center",

        font=dict(
            size=22,
            color="#263B5E"
        )
    ),

    xaxis=dict(
        title="Nearby Infrastructure Count →",
        rangemode="tozero",
        gridcolor="#EDF1F4",
        zeroline=False
    ),

    yaxis=dict(
        title="",
        showgrid=False,

        tickfont=dict(
            size=13,
            color="#263B5E"
        )
    ),

    plot_bgcolor="#FBFCFD",
    paper_bgcolor="white",

    width=1000,
    height=590,

    margin=dict(
        t=130,
        l=180,
        r=100,
        b=80
    )
)

fig.show()

In [ ]:
import networkx as nx
import plotly.graph_objects as go
import numpy as np

# ==========================================
# DARK STORE NEIGHBORHOOD WEB
# ==========================================

row = osm_proof.iloc[0]

brand = row["brand"]
activity_score = row["neighborhood_activity_score"]

# Combine OSM features into meaningful
# location-intelligence drivers
drivers = {
    "Residential\nBase": row["residential_areas"],
    "Commercial\nActivity": row["commercial_areas"],
    "Public\nTransport": row["public_transport_points"],
    "Education\nHubs": (
        row["nearby_schools"] +
        row["nearby_colleges"]
    ),
    "Healthcare": row["nearby_hospitals"],
    "Shopping\nHubs": row["nearby_malls"]
}


# ==========================================
# CREATE NETWORK
# ==========================================

G = nx.Graph()

center = "DARK STORE"

G.add_node(center)

for driver, value in drivers.items():

    G.add_node(
        driver,
        value=value
    )

    G.add_edge(
        center,
        driver,
        weight=max(value, 1)
    )


# ==========================================
# CUSTOM RADIAL POSITIONS
# ==========================================

positions = {
    center: (0, 0),

    "Residential\nBase": (-1.8, 0.9),
    "Commercial\nActivity": (1.8, 0.9),

    "Education\nHubs": (-2.0, -0.65),
    "Healthcare": (2.0, -0.65),

    "Public\nTransport": (-0.9, -1.65),
    "Shopping\nHubs": (0.9, -1.65)
}


# ==========================================
# FIGURE
# ==========================================

fig = go.Figure()


# ==========================================
# CONNECTIONS
# ==========================================

max_value = max(drivers.values())

for driver, value in drivers.items():

    x0, y0 = positions[center]
    x1, y1 = positions[driver]

    # Stronger environment = thicker connection
    width = (
        2 +
        (value / max_value) * 8
        if max_value > 0 else 2
    )

    fig.add_trace(
        go.Scatter(

            x=[x0, x1],
            y=[y0, y1],

            mode="lines",

            line=dict(
                width=width,
                color="rgba(72,96,120,0.25)"
            ),

            hoverinfo="skip",

            showlegend=False
        )
    )


# ==========================================
# OUTER NODES
# ==========================================

for driver, value in drivers.items():

    x, y = positions[driver]

    # Dynamic node size
    node_size = (
        35 +
        (value / max_value) * 45
        if max_value > 0 else 35
    )

    fig.add_trace(
        go.Scatter(

            x=[x],
            y=[y],

            mode="markers+text",

            marker=dict(

                size=node_size,

                color=value,

                colorscale=[
                    [0.00, "#D9F0E8"],
                    [0.35, "#73C6B6"],
                    [0.70, "#347C98"],
                    [1.00, "#313695"]
                ],

                cmin=0,
                cmax=max_value,

                line=dict(
                    color="white",
                    width=3
                ),

                showscale=False
            ),

            text=[
                f"<b>{driver}</b><br>{value}"
            ],

            textposition="top center",

            textfont=dict(
                size=12,
                color="#263B5E"
            ),

            hovertemplate=(
                f"<b>{driver.replace(chr(10),' ')}</b><br>"
                f"Nearby features: {value}"
                "<extra></extra>"
            ),

            showlegend=False
        )
    )


# ==========================================
# CENTER DARK STORE NODE
# ==========================================

fig.add_trace(
    go.Scatter(

        x=[0],
        y=[0],

        mode="markers+text",

        marker=dict(

            size=95,

            color="#172B4D",

            line=dict(
                color="white",
                width=5
            )
        ),

        text=[
            f"<b>DARK STORE</b><br>{brand}"
        ],

        textposition="middle center",

        textfont=dict(
            size=13,
            color="white"
        ),

        hovertemplate=(
            f"<b>{brand} Dark Store</b><br>"
            f"Neighborhood Activity Score: {activity_score}"
            "<extra></extra>"
        ),

        showlegend=False
    )
)


# ==========================================
# ACTIVITY SCORE BADGE
# ==========================================

fig.add_annotation(

    x=0,
    y=1.55,

    text=(
        "<b>NEIGHBORHOOD ACTIVITY SCORE</b><br>"
        f"<span style='font-size:20px'>{activity_score}</span>"
    ),

    showarrow=False,

    bgcolor="rgba(255,255,255,0.95)",

    bordercolor="#DDE4EA",
    borderwidth=1,
    borderpad=8,

    font=dict(
        color="#263B5E",
        size=11
    )
)


# ==========================================
# LAYOUT
# ==========================================

fig.update_layout(

    title=dict(

        text=(
            "<b>1 KM DARK-STORE NEIGHBORHOOD WEB</b><br>"
            "<span style='font-size:13px;color:#777777'>"
            "How the surrounding urban ecosystem supports "
            "micro-location suitability"
            "</span>"
        ),

        x=0.5,
        xanchor="center",

        font=dict(
            size=22,
            color="#263B5E"
        )
    ),

    xaxis=dict(
        visible=False,
        range=[-2.8, 2.8]
    ),

    yaxis=dict(
        visible=False,
        range=[-2.3, 2.2],
        scaleanchor="x",
        scaleratio=1
    ),

    plot_bgcolor="#FBFCFD",
    paper_bgcolor="white",

    width=1000,
    height=720,

    margin=dict(
        t=120,
        l=40,
        r=40,
        b=40
    ),

    showlegend=False
)

fig.show()